In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# =========================
# 0. Load files
# =========================
metab  = pd.read_csv(f"{BASE}/data/metabolomics/Metabolomics_maaslined_norun.csv")
immune = pd.read_csv(f"{BASE}/data/immuno/Immune_residue.csv")
quest  = pd.read_csv(f"{BASE}/data/quest_lab/Quest_residue.csv")
meta   = pd.read_csv(f"{BASE}/data/metadata/Metadata_061523.csv")
# =========================
# 1. Helper: transpose omics files
# rows = features, columns = samples
# we want rows = samples, columns = features
# =========================

def transpose_omics(df, prefix):
    df = df.copy()
    feature_col = df.columns[0]

    df = df.set_index(feature_col).T
    df.index.name = "sample_id"

    df.columns = [f"{prefix}__{str(c)}" for c in df.columns]
    df = df.reset_index()

    return df

metab_t = transpose_omics(metab, "metab")
immune_t = transpose_omics(immune, "immune")
quest_t = transpose_omics(quest, "quest")

# =========================
# 2. Metadata baseline only
# =========================

meta_tp1 = meta[meta["timepoints"] == "tp1"].copy()

meta_tp1 = meta_tp1.rename(columns={"Unnamed: 0": "sample_id"})

meta_tp1["y"] = meta_tp1["study_ptorhc"].map({
    "MECFS": 1,
    "HC": 0,
    "Control": 0
})

meta_tp1 = meta_tp1.dropna(subset=["y"])
meta_tp1["y"] = meta_tp1["y"].astype(int)

meta_small = meta_tp1[["sample_id", "study_ptorhc", "y"]].copy()

print("Metadata tp1:")
print(meta_small.shape)
print(meta_small["y"].value_counts())

# =========================
# 3. Merge modalities
# =========================

df = meta_small.merge(metab_t, on="sample_id", how="inner")
df = df.merge(immune_t, on="sample_id", how="inner")
df = df.merge(quest_t, on="sample_id", how="inner")

print("\nMerged multimodal df:")
print(df.shape)
print(df["y"].value_counts())

# =========================
# 4. Build X/y
# =========================

X_all = df.drop(columns=["sample_id", "study_ptorhc", "y"])
y = df["y"]

X_all = X_all.apply(pd.to_numeric, errors="coerce")
X_all = X_all.dropna(axis=1)

print("\nFinal X_all:")
print(X_all.shape)
print(y.value_counts())

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# =========================
# 1. Stable 15-metabolite panel
# =========================

X_metab = X_all[[c for c in X_all.columns if c.startswith("metab__")]]

n_runs = 100
selection_counts = pd.Series(0, index=X_metab.columns)

for seed in range(n_runs):
    X_train, X_test, y_train, y_test = train_test_split(
        X_metab,
        y,
        test_size=0.2,
        stratify=y,
        random_state=seed
    )

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=0.08,
            max_iter=5000,
            random_state=seed
        ))
    ])

    pipe.fit(X_train, y_train)

    coefs = pipe.named_steps["lasso"].coef_[0]
    selected = X_metab.columns[coefs != 0]
    selection_counts[selected] += 1

stable_metabs = selection_counts.sort_values(ascending=False)
top_metab_features = stable_metabs.head(15).index.tolist()

metab_panel_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=5000))
])

metab_panel_auc = cross_val_score(
    metab_panel_model,
    X_metab[top_metab_features],
    y,
    cv=cv,
    scoring="roc_auc"
)

# =========================
# 2. Metabolomics + immune
# =========================

X_metab_immune = X_all[
    [c for c in X_all.columns if c.startswith("metab__") or c.startswith("immune__")]
]

mi_model = Pipeline([
    ("scaler", StandardScaler()),
    ("elastic", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=0.1,
        l1_ratio=0.5,
        max_iter=10000,
        random_state=42
    ))
])

mi_auc = cross_val_score(
    mi_model,
    X_metab_immune,
    y,
    cv=cv,
    scoring="roc_auc"
)

# =========================
# 3. All modalities elastic net
# =========================

all_model = Pipeline([
    ("scaler", StandardScaler()),
    ("elastic", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=0.1,
        l1_ratio=0.5,
        max_iter=10000,
        random_state=42
    ))
])

all_auc = cross_val_score(
    all_model,
    X_all,
    y,
    cv=cv,
    scoring="roc_auc"
)

# =========================
# 4. Better models / upper bound
# =========================

rf_model = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=3,
    random_state=42,
    class_weight="balanced"
)

rf_auc = cross_val_score(
    rf_model,
    X_all,
    y,
    cv=cv,
    scoring="roc_auc"
)

svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        C=1,
        gamma="scale",
        probability=True,
        class_weight="balanced",
        random_state=42
    ))
])

svm_auc = cross_val_score(
    svm_model,
    X_all,
    y,
    cv=cv,
    scoring="roc_auc"
)

# =========================
# Results
# =========================

results = pd.DataFrame({
    "model": [
        "Stable 15-metabolite logistic",
        "Metabolomics + immune elastic net",
        "All modalities elastic net",
        "All modalities random forest",
        "All modalities SVM-RBF"
    ],
    "mean_auc": [
        metab_panel_auc.mean(),
        mi_auc.mean(),
        all_auc.mean(),
        rf_auc.mean(),
        svm_auc.mean()
    ],
    "std_auc": [
        metab_panel_auc.std(),
        mi_auc.std(),
        all_auc.std(),
        rf_auc.std(),
        svm_auc.std()
    ]
}).sort_values("mean_auc", ascending=False)

print("\nTop stable metabolites:")
for m in top_metab_features:
    print(m.replace("metab__", ""), "selected", stable_metabs[m], "/ 100 times")

print("\nAUC results:")
display(results)

Metadata tp1:
(249, 3)
1    153
0     96
Name: y, dtype: int64

Merged multimodal df:
(208, 1238)
1    131
0     77
Name: y, dtype: int64

Final X_all:
(208, 1235)
1    131
0     77
Name: y, dtype: int64

Top stable metabolites:
N4-acetylcytidine selected 100 / 100 times
N-acetylglutamate selected 99 / 100 times
3-(4-hydroxyphenyl)lactate selected 99 / 100 times
1-methylurate selected 89 / 100 times
N-palmitoyl-sphingosine (d18:1/16:0) selected 88 / 100 times
1-stearoyl-2-docosahexaenoyl-GPC (18:0/22:6) selected 86 / 100 times
X-11849 selected 80 / 100 times
vanillylmandelate (VMA) selected 78 / 100 times
glycodeoxycholate 3-sulfate selected 71 / 100 times
branched-chain, straight-chain, or cyclopropyl 10:1 fatty acid (1)* selected 70 / 100 times
sphingomyelin (d18:1/24:1, d18:2/24:0)* selected 68 / 100 times
X-25172 selected 64 / 100 times
4-vinylguaiacol sulfate selected 64 / 100 times
p-cresol glucuronide* selected 56 / 100 times
trigonelline (N'-methylnicotinate) selected 47 / 100 

,model,mean_auc,std_auc
0,Stable 15-metabolite logistic,0.897317,0.027577
3,All modalities random forest,0.814308,0.052113
2,All modalities elastic net,0.810717,0.056422
1,Metabolomics + immune elastic net,0.806890,0.054360
4,All modalities SVM-RBF,0.783222,0.021871


In [3]:
import os

for root, dirs, files in os.walk("."):
    if "Metabolomics_maaslined_norun.csv" in files:
        print(os.path.join(root, "Metabolomics_maaslined_norun.csv"))

In [5]:
from pathlib import Path
import pandas as pd

def find_file(filename):
    matches = list(Path.cwd().parent.rglob(filename)) + list(Path.cwd().rglob(filename))
    matches = list(dict.fromkeys(matches))
    
    if len(matches) == 0:
        raise FileNotFoundError(f"Could not find {filename}. Current folder: {Path.cwd()}")
    
    print(filename, "->", matches[0])
    return matches[0]

metab  = pd.read_csv(find_file("Metabolomics_maaslined_norun.csv"))
immune = pd.read_csv(find_file("Immune_residue.csv"))
quest  = pd.read_csv(find_file("Quest_residue.csv"))
meta   = pd.read_csv(find_file("Metadata_061523.csv"))

print("Loaded:")
print("metab:", metab.shape)
print("immune:", immune.shape)
print("quest:", quest.shape)
print("meta:", meta.shape)

FileNotFoundError: Could not find Metabolomics_maaslined_norun.csv. Current folder: c:\Users\rohan\OneDrive\Desktop\ME-CFS\src\main

In [7]:
BASE = "../.."

metab  = pd.read_csv(f"{BASE}/data/metabolomics/Metabolomics_maaslined_norun.csv")
immune = pd.read_csv(f"{BASE}/data/immuno/Immune_residue.csv")
quest  = pd.read_csv(f"{BASE}/data/quest_lab/Quest_residue.csv")
meta   = pd.read_csv(f"{BASE}/data/metadata/Metadata_061523.csv")

print(metab.shape)
print(immune.shape)
print(quest.shape)
print(meta.shape)

(876, 415)
(311, 490)
(48, 504)
(515, 22)


In [9]:
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix

# =========================
# TRUE HELD-OUT TEST
# Feature selection happens on TRAIN ONLY
# =========================

X_metab = X_all[[c for c in X_all.columns if c.startswith("metab__")]]

X_train, X_test, y_train, y_test = train_test_split(
    X_metab,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# stable LASSO feature selection on TRAIN ONLY
n_runs = 100
selection_counts_train = pd.Series(0, index=X_train.columns)

for seed in range(n_runs):
    X_subtrain, X_val, y_subtrain, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        stratify=y_train,
        random_state=seed
    )

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=0.08,
            max_iter=5000,
            random_state=seed
        ))
    ])

    pipe.fit(X_subtrain, y_subtrain)

    coefs = pipe.named_steps["lasso"].coef_[0]
    selected = X_train.columns[coefs != 0]
    selection_counts_train[selected] += 1

stable_train_metabs = selection_counts_train.sort_values(ascending=False)
heldout_features = stable_train_metabs.head(15).index.tolist()

print("Train-only selected metabolites:")
for f in heldout_features:
    print(f.replace("metab__", ""), "selected", stable_train_metabs[f], "/ 100 times")

# final model trained ONLY on train
heldout_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=5000))
])

heldout_model.fit(X_train[heldout_features], y_train)

y_prob = heldout_model.predict_proba(X_test[heldout_features])[:, 1]
y_pred = heldout_model.predict(X_test[heldout_features])

auc = roc_auc_score(y_test, y_prob)
acc = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\nHeld-out test performance:")
print("AUC:", auc)
print("Accuracy:", acc)
print("Balanced accuracy:", bal_acc)
print("\nConfusion matrix:")
print(cm)

Train-only selected metabolites:
N-acetylglutamate selected 100 / 100 times
1-methylurate selected 100 / 100 times
N-palmitoyl-sphingosine (d18:1/16:0) selected 81 / 100 times
3-(4-hydroxyphenyl)lactate selected 70 / 100 times
N4-acetylcytidine selected 60 / 100 times
glycodeoxycholate 3-sulfate selected 56 / 100 times
1-stearoyl-2-arachidonoyl-GPC (18:0/20:4) selected 48 / 100 times
4-methylcatechol sulfate selected 48 / 100 times
phenyllactate (PLA) selected 47 / 100 times
X-18886 selected 39 / 100 times
4-methylguaiacol sulfate selected 38 / 100 times
p-cresol glucuronide* selected 38 / 100 times
4-vinylguaiacol sulfate selected 34 / 100 times
N-palmitoyl-sphinganine (d18:0/16:0) selected 33 / 100 times
alpha-tocopherol selected 32 / 100 times

Held-out test performance:
AUC: 0.7548076923076923
Accuracy: 0.7619047619047619
Balanced accuracy: 0.7235576923076923

Confusion matrix:
[[ 9  7]
 [ 3 23]]


In [10]:
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.impute import SimpleImputer

def heldout_eval(X, y, name):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    models = {
        "Elastic Net": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                C=0.1,
                l1_ratio=0.5,
                max_iter=20000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=500,
                min_samples_leaf=3,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        "SVM-RBF": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(
                kernel="rbf",
                C=1,
                gamma="scale",
                probability=True,
                class_weight="balanced",
                random_state=42
            ))
        ])
    }

    out = []

    for model_name, model in models.items():
        model.fit(X_train, y_train)

        y_prob = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)

        out.append({
            "dataset": name,
            "model": model_name,
            "n_features": X.shape[1],
            "auc": roc_auc_score(y_test, y_prob),
            "accuracy": accuracy_score(y_test, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
            "confusion_matrix": confusion_matrix(y_test, y_pred)
        })

    return out


X_metab = X_all[[c for c in X_all.columns if c.startswith("metab__")]]
X_immune = X_all[[c for c in X_all.columns if c.startswith("immune__")]]
X_quest = X_all[[c for c in X_all.columns if c.startswith("quest__")]]

dataset_dict = {
    "Metabolomics only": X_metab,
    "Immune only": X_immune,
    "Quest only": X_quest,
    "Metabolomics + immune": pd.concat([X_metab, X_immune], axis=1),
    "Metabolomics + quest": pd.concat([X_metab, X_quest], axis=1),
    "Immune + quest": pd.concat([X_immune, X_quest], axis=1),
    "All datasets": pd.concat([X_metab, X_immune, X_quest], axis=1)
}

all_results = []

for name, X in dataset_dict.items():
    print("Running:", name, X.shape)
    all_results.extend(heldout_eval(X, y, name))

results_heldout_all = pd.DataFrame(all_results)

results_summary = results_heldout_all[
    ["dataset", "model", "n_features", "auc", "accuracy", "balanced_accuracy"]
].sort_values("auc", ascending=False)

display(results_summary)

print("\nConfusion matrices:")
for _, row in results_heldout_all.sort_values("auc", ascending=False).iterrows():
    print("\n", row["dataset"], "-", row["model"])
    print("AUC:", row["auc"])
    print(row["confusion_matrix"])

Running: Metabolomics only (208, 876)
Running: Immune only (208, 311)
Running: Quest only (208, 48)
Running: Metabolomics + immune (208, 1187)
Running: Metabolomics + quest (208, 924)
Running: Immune + quest (208, 359)
Running: All datasets (208, 1235)


,dataset,model,n_features,auc,accuracy,balanced_accuracy
10,Metabolomics + immune,Random Forest,1187,0.822115,0.714286,0.637019
1,Metabolomics only,Random Forest,876,0.822115,0.809524,0.750000
11,Metabolomics + immune,SVM-RBF,1187,0.814904,0.833333,0.781250
13,Metabolomics + quest,Random Forest,924,0.798077,0.809524,0.750000
12,Metabolomics + quest,Elastic Net,924,0.795673,0.809524,0.774038
20,All datasets,SVM-RBF,1235,0.794471,0.785714,0.730769
19,All datasets,Random Forest,1235,0.790865,0.785714,0.718750
18,All datasets,Elastic Net,1235,0.788462,0.666667,0.658654
5,Immune only,SVM-RBF,311,0.769231,0.809524,0.798077
2,Metabolomics only,SVM-RBF,876,0.759615,0.714286,0.685096



Confusion matrices:

 Metabolomics + immune - Random Forest
AUC: 0.8221153846153846
[[ 5 11]
 [ 1 25]]

 Metabolomics only - Random Forest
AUC: 0.8221153846153846
[[ 8  8]
 [ 0 26]]

 Metabolomics + immune - SVM-RBF
AUC: 0.8149038461538461
[[ 9  7]
 [ 0 26]]

 Metabolomics + quest - Random Forest
AUC: 0.7980769230769231
[[ 8  8]
 [ 0 26]]

 Metabolomics + quest - Elastic Net
AUC: 0.7956730769230769
[[10  6]
 [ 2 24]]

 All datasets - SVM-RBF
AUC: 0.7944711538461539
[[ 8  8]
 [ 1 25]]

 All datasets - Random Forest
AUC: 0.7908653846153846
[[ 7  9]
 [ 0 26]]

 All datasets - Elastic Net
AUC: 0.7884615384615385
[[10  6]
 [ 8 18]]

 Immune only - SVM-RBF
AUC: 0.7692307692307693
[[12  4]
 [ 4 22]]

 Metabolomics only - SVM-RBF
AUC: 0.7596153846153846
[[ 9  7]
 [ 5 21]]

 Immune + quest - SVM-RBF
AUC: 0.7596153846153846
[[10  6]
 [ 2 24]]

 Metabolomics + immune - Elastic Net
AUC: 0.7572115384615384
[[ 9  7]
 [ 8 18]]

 Metabolomics only - Elastic Net
AUC: 0.7572115384615384
[[10  6]
 [ 2 2

^C
Note: you may need to restart the kernel to use updated packages.


   ---------------------------------------- 0.0/124.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.9 MB ? eta -:--:--
   ---------------------------------------- 1.0/124.9 MB 3.4 MB/s eta 0:00:37
   ---------------------------------------- 1.3/124.9 MB 3.7 MB/s eta 0:00:34
    --------------------------------------- 1.8/124.9 MB 2.4 MB/s eta 0:00:52
   - -------------------------------------- 3.9/124.9 MB 4.3 MB/s eta 0:00:29
   -- ------------------------------------- 6.6/124.9 MB 5.8 MB/s eta 0:00:21
   --- ------------------------------------ 9.4/124.9 MB 7.2 MB/s eta 0:00:17
   ---- ----------------------------------- 14.4/124.9 MB 9.4 MB/s eta 0:00:12
   ------ --------------------------------- 20.4/124.9 MB 11.7 MB/s eta 0:00:09
   -------- ------------------------------- 26.7/124.9 MB 13.8 MB/s eta 0:00:08
   ---------- ----------------------------- 34.1/124.9 MB 15.8 MB/s eta 0:00:06
   ------------- -------------------------- 41.2/124.9 MB 17.2 MB/s eta


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

# Metabolomics only
X_metab = X_all[[c for c in X_all.columns if c.startswith("metab__")]]

X_train, X_test, y_train, y_test = train_test_split(
    X_metab,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

y_prob = xgb.predict_proba(X_test)[:, 1]
y_pred = xgb.predict(X_test)

auc = roc_auc_score(y_test, y_prob)
acc = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("=== XGBoost: Metabolomics Only ===")
print("AUC:", auc)
print("Accuracy:", acc)
print("Balanced Accuracy:", bal_acc)
print("\nConfusion Matrix:")
print(cm)

# Feature importance
importance = pd.Series(
    xgb.feature_importances_,
    index=X_metab.columns
).sort_values(ascending=False)

print("\nTop 20 Features:")
display(importance.head(20))

ValueError: feature_names must be string, and may not contain [, ] or <

In [13]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

X_metab = X_all[[c for c in X_all.columns if c.startswith("metab__")]]

# Rename columns safely for XGBoost
safe_cols = [f"f{i}" for i in range(X_metab.shape[1])]
feature_map = dict(zip(safe_cols, X_metab.columns))

X_metab_safe = X_metab.copy()
X_metab_safe.columns = safe_cols

X_train, X_test, y_train, y_test = train_test_split(
    X_metab_safe,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

y_prob = xgb.predict_proba(X_test)[:, 1]
y_pred = xgb.predict(X_test)

print("=== XGBoost: Metabolomics Only ===")
print("AUC:", roc_auc_score(y_test, y_prob))
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

importance = pd.Series(
    xgb.feature_importances_,
    index=X_metab_safe.columns
).sort_values(ascending=False)

importance_real_names = importance.rename(index=feature_map)

print("\nTop 20 Features:")
display(importance_real_names.head(20))

=== XGBoost: Metabolomics Only ===
AUC: 0.8269230769230769
Accuracy: 0.7619047619047619
Balanced Accuracy: 0.7355769230769231

Confusion Matrix:
[[10  6]
 [ 4 22]]

Top 20 Features:


metab__metabolonic lactone sulfate                          0.015791
metab__dihomo-linoleoylcarnitine (C20:2)*                   0.014833
metab__pyridoxate                                           0.012537
metab__X-17654                                              0.011876
metab__palmitoyl dihydrosphingomyelin (d18:0/16:0)*         0.011542
metab__caffeic acid sulfate                                 0.010812
metab__cinnamoylglycine                                     0.010615
metab__X-21829                                              0.010052
metab__4-hydroxyphenylacetate                               0.009875
metab__1,2-dilinoleoyl-GPC (18:2/18:2)                      0.009763
metab__octanoylcarnitine (C8)                               0.008403
metab__2'-deoxyuridine                                      0.008295
metab__indolepropionate                                     0.008274
metab__aconitate [cis or trans]                             0.008095
metab__imidazole propionate       

In [15]:
# rebuild original metabolomics matrix
X_metab = X_all[[c for c in X_all.columns if c.startswith("metab__")]]

# split ORIGINAL names
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_metab,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

score_features = stable_train_metabs.head(5).index.tolist()

score_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=5000))
])

score_model.fit(X_train_m[score_features], y_train_m)

coefs = score_model.named_steps["logreg"].coef_[0]
intercept = score_model.named_steps["logreg"].intercept_[0]

score_table = pd.DataFrame({
    "metabolite": [f.replace("metab__", "") for f in score_features],
    "coefficient": coefs
})

print("Intercept:", intercept)
display(score_table)

y_prob_score = score_model.predict_proba(X_test_m[score_features])[:, 1]
y_pred_score = score_model.predict(X_test_m[score_features])

print("5-metabolite score held-out AUC:", roc_auc_score(y_test_m, y_prob_score))
print("Accuracy:", accuracy_score(y_test_m, y_pred_score))
print("Balanced accuracy:", balanced_accuracy_score(y_test_m, y_pred_score))
print(confusion_matrix(y_test_m, y_pred_score))

Intercept: 0.7928788570930335


,metabolite,coefficient
0,N-acetylglutamate,0.890762
1,1-methylurate,-0.711126
2,N-palmitoyl-sphingosine (d18:1/16:0),0.794240
3,3-(4-hydroxyphenyl)lactate,-0.799580
4,N4-acetylcytidine,0.670743


5-metabolite score held-out AUC: 0.7451923076923077
Accuracy: 0.6904761904761905
Balanced accuracy: 0.6658653846153846
[[ 9  7]
 [ 6 20]]


In [16]:
yesterday_panel = [
    "metab__1-stearoyl-2-docosahexaenoyl-GPC (18:0/22:6)",
    "metab__3-(4-hydroxyphenyl)lactate",
    "metab__N-acetylglutamate",
    "metab__N-palmitoyl-sphinganine (d18:0/16:0)",
    "metab__N-palmitoyl-sphingosine (d18:1/16:0)"
]

missing = [f for f in yesterday_panel if f not in X_metab.columns]
print("Missing:", missing)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_metab,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

yesterday_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=5000))
])

yesterday_model.fit(X_train_m[yesterday_panel], y_train_m)

coefs = yesterday_model.named_steps["logreg"].coef_[0]
intercept = yesterday_model.named_steps["logreg"].intercept_[0]

yesterday_score_table = pd.DataFrame({
    "metabolite": [f.replace("metab__", "") for f in yesterday_panel],
    "coefficient": coefs
})

print("Intercept:", intercept)
display(yesterday_score_table)

y_prob = yesterday_model.predict_proba(X_test_m[yesterday_panel])[:, 1]
y_pred = yesterday_model.predict(X_test_m[yesterday_panel])

print("Yesterday 5-panel held-out AUC:", roc_auc_score(y_test_m, y_prob))
print("Accuracy:", accuracy_score(y_test_m, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test_m, y_pred))
print(confusion_matrix(y_test_m, y_pred))

Missing: []
Intercept: 0.7536671832738141


,metabolite,coefficient
0,1-stearoyl-2-docosahexaenoyl-GPC (18:0/22:6),0.245246
1,3-(4-hydroxyphenyl)lactate,-0.807760
2,N-acetylglutamate,0.712386
3,N-palmitoyl-sphinganine (d18:0/16:0),0.276531
4,N-palmitoyl-sphingosine (d18:1/16:0),0.704586


Yesterday 5-panel held-out AUC: 0.7548076923076923
Accuracy: 0.7857142857142857
Balanced accuracy: 0.7427884615384616
[[ 9  7]
 [ 2 24]]


In [17]:
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix

# =========================
# Reproducible train-only LASSO + RF overlap panel
# =========================

X_metab = X_all[[c for c in X_all.columns if c.startswith("metab__")]]

X_train, X_test, y_train, y_test = train_test_split(
    X_metab,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# -------------------------
# LASSO on train only
# -------------------------

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

lasso = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=0.06,
    max_iter=5000,
    random_state=42
)

lasso.fit(X_train_scaled, y_train)

lasso_coefs = pd.Series(
    lasso.coef_[0],
    index=X_train.columns
)

lasso_selected = lasso_coefs[lasso_coefs != 0].sort_values(
    key=abs,
    ascending=False
)

# -------------------------
# Random Forest on train only
# -------------------------

rf = RandomForestClassifier(
    n_estimators=1000,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

rf_importance = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

rf_top20 = rf_importance.head(20)

# -------------------------
# Overlap panel
# -------------------------

panel = list(lasso_selected.index.intersection(rf_top20.index))

print("LASSO selected:", len(lasso_selected))
display(lasso_selected)

print("\nRF top 20:")
display(rf_top20)

print("\nTrain-only overlap panel:")
for p in panel:
    print(p.replace("metab__", ""))

print("\nNumber of metabolites:", len(panel))

# -------------------------
# Test panel on held-out set
# -------------------------

panel_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=5000))
])

panel_model.fit(X_train[panel], y_train)

y_prob = panel_model.predict_proba(X_test[panel])[:, 1]
y_pred = panel_model.predict(X_test[panel])

print("\nHeld-out panel performance:")
print("AUC:", roc_auc_score(y_test, y_prob))
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

coefs = panel_model.named_steps["logreg"].coef_[0]
intercept = panel_model.named_steps["logreg"].intercept_[0]

score_table = pd.DataFrame({
    "metabolite": [p.replace("metab__", "") for p in panel],
    "coefficient": coefs
})

print("\nIntercept:", intercept)
display(score_table)

LASSO selected: 13


metab__1-methylurate                               -0.192617
metab__N-acetylglutamate                            0.187168
metab__N-palmitoyl-sphingosine (d18:1/16:0)         0.166762
metab__3-(4-hydroxyphenyl)lactate                  -0.067126
metab__N4-acetylcytidine                            0.052567
metab__4-methylguaiacol sulfate                     0.021316
metab__phenyllactate (PLA)                         -0.015355
metab__glycodeoxycholate 3-sulfate                  0.013666
metab__1-stearoyl-2-arachidonoyl-GPC (18:0/20:4)    0.013358
metab__4-methylcatechol sulfate                     0.012736
metab__p-cresol glucuronide*                        0.009057
metab__N-stearoyl-sphinganine (d18:0/18:0)*         0.007504
metab__sphingomyelin (d18:1/18:1, d18:2/18:0)       0.001740
dtype: float64


RF top 20:


metab__X-11849                                      0.013323
metab__X-11847                                      0.011899
metab__N-acetylglutamate                            0.009775
metab__3-(4-hydroxyphenyl)lactate                   0.007631
metab__p-cresol glucuronide*                        0.007590
metab__5-(galactosylhydroxy)-lysine                 0.007483
metab__X-11858                                      0.006820
metab__alpha-tocopherol                             0.006482
metab__N-palmitoyl-sphingosine (d18:1/16:0)         0.006393
metab__X-11795                                      0.006339
metab__1-methylurate                                0.006012
metab__N4-acetylcytidine                            0.005873
metab__beta-cryptoxanthin                           0.005868
metab__N-palmitoyl-sphinganine (d18:0/16:0)         0.005811
metab__X-11372                                      0.005734
metab__phenyllactate (PLA)                          0.005621
metab__2-naphthol sulfat


Train-only overlap panel:
1-methylurate
N-acetylglutamate
N-palmitoyl-sphingosine (d18:1/16:0)
3-(4-hydroxyphenyl)lactate
N4-acetylcytidine
phenyllactate (PLA)
1-stearoyl-2-arachidonoyl-GPC (18:0/20:4)
p-cresol glucuronide*

Number of metabolites: 8

Held-out panel performance:
AUC: 0.7475961538461537
Accuracy: 0.6904761904761905
Balanced accuracy: 0.6658653846153846
[[ 9  7]
 [ 6 20]]

Intercept: 0.9888662172976056


,metabolite,coefficient
0,1-methylurate,-0.922929
1,N-acetylglutamate,0.871537
2,N-palmitoyl-sphingosine (d18:1/16:0),0.473737
3,3-(4-hydroxyphenyl)lactate,-0.564145
4,N4-acetylcytidine,0.635293
5,phenyllactate (PLA),-0.374276
6,1-stearoyl-2-arachidonoyl-GPC (18:0/20:4),0.325600
7,p-cresol glucuronide*,1.041142


In [19]:
from collections import Counter
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix

# =====================================
# Fixed held-out test set
# =====================================

X_metab = X_all[[c for c in X_all.columns if c.startswith("metab__")]]

X_train, X_test, y_train, y_test = train_test_split(
    X_metab,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# =====================================
# Stability selection on TRAIN ONLY
# =====================================

selection_counter = Counter()

sss = StratifiedShuffleSplit(
    n_splits=100,
    test_size=0.2,
    random_state=42
)

for train_idx, _ in sss.split(X_train, y_train):

    X_sub = X_train.iloc[train_idx]
    y_sub = y_train.iloc[train_idx]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=0.06,
            max_iter=5000,
            random_state=42
        ))
    ])

    model.fit(X_sub, y_sub)

    coefs = model.named_steps["lasso"].coef_[0]

    selected = X_sub.columns[coefs != 0]

    for feat in selected:
        selection_counter[feat] += 1

# =====================================
# Selection frequencies
# =====================================

stability_df = pd.DataFrame({
    "metabolite": list(selection_counter.keys()),
    "count": list(selection_counter.values())
})

stability_df = stability_df.sort_values(
    "count",
    ascending=False
)

stability_df["frequency"] = stability_df["count"] / 100

display(stability_df.head(25))

# =====================================
# Top 5 most stable metabolites
# =====================================

stable_panel = stability_df.head(8)["metabolite"].tolist()

print("\nStable panel:")
for m in stable_panel:
    print(m.replace("metab__", ""))

# =====================================
# Final score model
# =====================================

score_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=5000))
])

score_model.fit(
    X_train[stable_panel],
    y_train
)

y_prob = score_model.predict_proba(
    X_test[stable_panel]
)[:, 1]

y_pred = score_model.predict(
    X_test[stable_panel]
)

print("\nHeld-out performance")
print("AUC:", roc_auc_score(y_test, y_prob))
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# =====================================
# Final score formula
# =====================================

coefs = score_model.named_steps["logreg"].coef_[0]
intercept = score_model.named_steps["logreg"].intercept_[0]

formula_df = pd.DataFrame({
    "metabolite": [m.replace("metab__", "") for m in stable_panel],
    "coefficient": coefs
})

print("\nIntercept:", intercept)
display(formula_df)

,metabolite,count,frequency
2,metab__1-methylurate,91,0.91
0,metab__N-acetylglutamate,82,0.82
5,metab__N-palmitoyl-sphingosine (d18:1/16:0),80,0.80
16,metab__N-palmitoyl-sphinganine (d18:0/16:0),22,0.22
12,metab__1-stearoyl-2-arachidonoyl-GPC (18:0/20:4),21,0.21
4,metab__N-stearoyl-sphinganine (d18:0/18:0)*,20,0.20
6,metab__phenyllactate (PLA),20,0.20
15,metab__3-(4-hydroxyphenyl)lactate,20,0.20
7,"metab__sphingomyelin (d18:1/18:1, d18:2/18:0)",19,0.19
11,metab__imidazole lactate,18,0.18



Stable panel:
1-methylurate
N-acetylglutamate
N-palmitoyl-sphingosine (d18:1/16:0)
N-palmitoyl-sphinganine (d18:0/16:0)
1-stearoyl-2-arachidonoyl-GPC (18:0/20:4)
N-stearoyl-sphinganine (d18:0/18:0)*
phenyllactate (PLA)
3-(4-hydroxyphenyl)lactate

Held-out performance
AUC: 0.6899038461538461
Accuracy: 0.7380952380952381
Balanced accuracy: 0.7043269230769231
[[ 9  7]
 [ 4 22]]

Intercept: 0.767819036780669


,metabolite,coefficient
0,1-methylurate,-0.740908
1,N-acetylglutamate,0.915032
2,N-palmitoyl-sphingosine (d18:1/16:0),0.435089
3,N-palmitoyl-sphinganine (d18:0/16:0),0.012951
4,1-stearoyl-2-arachidonoyl-GPC (18:0/20:4),0.427019
5,N-stearoyl-sphinganine (d18:0/18:0)*,0.449524
6,phenyllactate (PLA),-0.400969
7,3-(4-hydroxyphenyl)lactate,-0.610978


In [20]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score

# Same train/test split as before
X_metab = X_all[[c for c in X_all.columns if c.startswith("metab__")]]

X_train, X_test, y_train, y_test = train_test_split(
    X_metab,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

results = []

for k in [5, 10, 15, 20, 25, 30]:

    panel = stability_df.head(k)["metabolite"].tolist()

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=5000))
    ])

    model.fit(X_train[panel], y_train)

    y_prob = model.predict_proba(X_test[panel])[:, 1]
    y_pred = model.predict(X_test[panel])

    results.append({
        "n_metabolites": k,
        "auc": roc_auc_score(y_test, y_prob),
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)

display(results_df.sort_values("auc", ascending=False))

,n_metabolites,auc,accuracy,balanced_accuracy
5,30,0.742788,0.690476,0.641827
4,25,0.737981,0.690476,0.641827
2,15,0.730769,0.690476,0.653846
3,20,0.728365,0.714286,0.673077
1,10,0.692308,0.738095,0.704327
0,5,0.663462,0.666667,0.646635


In [21]:
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import numpy as np

aucs = []

for seed in range(100):

    X_train, X_test, y_train, y_test = train_test_split(
        X_metab,
        y,
        test_size=0.2,
        stratify=y,
        random_state=seed
    )

    # LASSO train only
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=0.06,
            max_iter=5000
        ))
    ])

    pipe.fit(X_train, y_train)

    coefs = pipe.named_steps["lasso"].coef_[0]

    selected = X_train.columns[coefs != 0]

    # skip tiny panels
    if len(selected) < 5:
        continue

    panel = selected[:min(15, len(selected))]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=5000))
    ])

    model.fit(X_train[panel], y_train)

    probs = model.predict_proba(X_test[panel])[:,1]

    aucs.append(
        roc_auc_score(y_test, probs)
    )

print("Mean AUC:", np.mean(aucs))
print("Median AUC:", np.median(aucs))
print("Std:", np.std(aucs))
print("Min:", np.min(aucs))
print("Max:", np.max(aucs))

Mean AUC: 0.747331730769231
Median AUC: 0.7548076923076923
Std: 0.06329185411371961
Min: 0.59375
Max: 0.9038461538461539


In [22]:
yesterday_panel = [
    "metab__1-stearoyl-2-docosahexaenoyl-GPC (18:0/22:6)",
    "metab__3-(4-hydroxyphenyl)lactate",
    "metab__N-acetylglutamate",
    "metab__N-palmitoyl-sphinganine (d18:0/16:0)",
    "metab__N-palmitoyl-sphingosine (d18:1/16:0)"
]

aucs = []

for seed in range(100):
    X_train, X_test, y_train, y_test = train_test_split(
        X_metab,
        y,
        test_size=0.2,
        stratify=y,
        random_state=seed
    )

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=5000))
    ])

    model.fit(X_train[yesterday_panel], y_train)

    probs = model.predict_proba(X_test[yesterday_panel])[:, 1]
    aucs.append(roc_auc_score(y_test, probs))

print("Yesterday exact 5-panel repeated split:")
print("Mean AUC:", np.mean(aucs))
print("Median AUC:", np.median(aucs))
print("Std:", np.std(aucs))
print("Min:", np.min(aucs))
print("Max:", np.max(aucs))

Yesterday exact 5-panel repeated split:
Mean AUC: 0.8031730769230769
Median AUC: 0.8040865384615385
Std: 0.07052843649925601
Min: 0.5360576923076923
Max: 0.9399038461538461


In [23]:
# Check where yesterday panel ranks in stability selection
yesterday_panel = [
    "metab__1-stearoyl-2-docosahexaenoyl-GPC (18:0/22:6)",
    "metab__3-(4-hydroxyphenyl)lactate",
    "metab__N-acetylglutamate",
    "metab__N-palmitoyl-sphinganine (d18:0/16:0)",
    "metab__N-palmitoyl-sphingosine (d18:1/16:0)"
]

ranked_stability = stability_df.reset_index(drop=True).copy()
ranked_stability["rank"] = ranked_stability.index + 1

ranked_stability["clean_name"] = ranked_stability["metabolite"].str.replace("metab__", "", regex=False)

display(
    ranked_stability[
        ranked_stability["metabolite"].isin(yesterday_panel)
    ][["rank", "clean_name", "count", "frequency"]]
)

display(ranked_stability.head(30)[["rank", "clean_name", "count", "frequency"]])

,rank,clean_name,count,frequency
1,2,N-acetylglutamate,82,0.82
2,3,N-palmitoyl-sphingosine (d18:1/16:0),80,0.80
3,4,N-palmitoyl-sphinganine (d18:0/16:0),22,0.22
7,8,3-(4-hydroxyphenyl)lactate,20,0.20
40,41,1-stearoyl-2-docosahexaenoyl-GPC (18:0/22:6),1,0.01


,rank,clean_name,count,frequency
0,1,1-methylurate,91,0.91
1,2,N-acetylglutamate,82,0.82
2,3,N-palmitoyl-sphingosine (d18:1/16:0),80,0.80
3,4,N-palmitoyl-sphinganine (d18:0/16:0),22,0.22
4,5,1-stearoyl-2-arachidonoyl-GPC (18:0/20:4),21,0.21
5,6,N-stearoyl-sphinganine (d18:0/18:0)*,20,0.20
6,7,phenyllactate (PLA),20,0.20
7,8,3-(4-hydroxyphenyl)lactate,20,0.20
8,9,"sphingomyelin (d18:1/18:1, d18:2/18:0)",19,0.19
9,10,imidazole lactate,18,0.18


In [24]:
results = []

for k in range(4, 16):

    panel = stability_df.head(k)["metabolite"].tolist()

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=5000))
    ])

    model.fit(X_train[panel], y_train)

    probs = model.predict_proba(X_test[panel])[:, 1]
    preds = model.predict(X_test[panel])

    results.append({
        "k": k,
        "auc": roc_auc_score(y_test, probs),
        "accuracy": accuracy_score(y_test, preds),
        "balanced_accuracy": balanced_accuracy_score(y_test, preds)
    })

results_df = pd.DataFrame(results)

display(results_df.sort_values("auc", ascending=False))

,k,auc,accuracy,balanced_accuracy
0,4,0.829327,0.714286,0.661058
4,8,0.817308,0.785714,0.754808
7,11,0.817308,0.809524,0.774038
8,12,0.814904,0.809524,0.774038
9,13,0.802885,0.833333,0.805288
10,14,0.802885,0.833333,0.805288
11,15,0.800481,0.809524,0.774038
5,9,0.793269,0.761905,0.723558
6,10,0.790865,0.761905,0.723558
3,7,0.790865,0.690476,0.653846


In [25]:
top4_panel = stability_df.head(4)["metabolite"].tolist()

aucs = []

for seed in range(100):

    X_train, X_test, y_train, y_test = train_test_split(
        X_metab,
        y,
        test_size=0.2,
        stratify=y,
        random_state=seed
    )

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=5000))
    ])

    model.fit(X_train[top4_panel], y_train)

    probs = model.predict_proba(X_test[top4_panel])[:,1]

    aucs.append(
        roc_auc_score(y_test, probs)
    )

print("Mean:", np.mean(aucs))
print("Median:", np.median(aucs))
print("Std:", np.std(aucs))
print("Min:", np.min(aucs))
print("Max:", np.max(aucs))

Mean: 0.7654326923076922
Median: 0.7704326923076923
Std: 0.07025417123689832
Min: 0.5432692307692308
Max: 0.9230769230769231


In [26]:
results = []

for k in range(4, 16):

    panel = stability_df.head(k)["metabolite"].tolist()

    aucs = []

    for seed in range(100):

        X_train, X_test, y_train, y_test = train_test_split(
            X_metab,
            y,
            test_size=0.2,
            stratify=y,
            random_state=seed
        )

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("logreg", LogisticRegression(max_iter=5000))
        ])

        model.fit(X_train[panel], y_train)

        probs = model.predict_proba(X_test[panel])[:, 1]

        aucs.append(
            roc_auc_score(y_test, probs)
        )

    results.append({
        "k": k,
        "mean_auc": np.mean(aucs),
        "median_auc": np.median(aucs),
        "std_auc": np.std(aucs),
        "min_auc": np.min(aucs),
        "max_auc": np.max(aucs)
    })

results = pd.DataFrame(results)

display(
    results.sort_values("mean_auc", ascending=False)
)

,k,mean_auc,median_auc,std_auc,min_auc,max_auc
11,15,0.830409,0.838942,0.057072,0.646635,0.932692
9,13,0.818341,0.824519,0.061048,0.629808,0.956731
10,14,0.817957,0.823317,0.059896,0.622596,0.935096
7,11,0.811947,0.817308,0.066449,0.596154,0.947115
8,12,0.810601,0.817308,0.064893,0.600962,0.947115
4,8,0.810120,0.817308,0.069355,0.579327,0.959135
5,9,0.806514,0.811298,0.068462,0.579327,0.937500
6,10,0.801370,0.805288,0.068690,0.584135,0.937500
3,7,0.790385,0.802885,0.067978,0.581731,0.927885
0,4,0.765433,0.770433,0.070254,0.543269,0.923077


In [29]:
top15_panel = stability_df.head(15)["metabolite"].tolist()

print("Top 15 stability-selected metabolites:\n")

for i, m in enumerate(top15_panel, start=1):
    print(f"{i:2d}. {m.replace('metab__', '')}")

Top 15 stability-selected metabolites:

 1. 1-methylurate
 2. N-acetylglutamate
 3. N-palmitoyl-sphingosine (d18:1/16:0)
 4. N-palmitoyl-sphinganine (d18:0/16:0)
 5. 1-stearoyl-2-arachidonoyl-GPC (18:0/20:4)
 6. N-stearoyl-sphinganine (d18:0/18:0)*
 7. phenyllactate (PLA)
 8. 3-(4-hydroxyphenyl)lactate
 9. sphingomyelin (d18:1/18:1, d18:2/18:0)
10. imidazole lactate
11. N4-acetylcytidine
12. alpha-tocopherol
13. 4-methylcatechol sulfate
14. 4-vinylguaiacol sulfate
15. glycodeoxycholate 3-sulfate


In [30]:
from itertools import combinations
from collections import Counter
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# =========================
# 1. Fixed train/test split
# =========================

X_metab = X_all[[c for c in X_all.columns if c.startswith("metab__")]]

X_train, X_test, y_train, y_test = train_test_split(
    X_metab,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# =========================
# 2. Build stable candidate pool on TRAIN ONLY
# =========================

counter = Counter()

sss = StratifiedShuffleSplit(
    n_splits=200,
    test_size=0.2,
    random_state=42
)

for idx, _ in sss.split(X_train, y_train):
    X_sub = X_train.iloc[idx]
    y_sub = y_train.iloc[idx]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=0.06,
            max_iter=5000,
            random_state=42
        ))
    ])

    model.fit(X_sub, y_sub)

    coefs = model.named_steps["lasso"].coef_[0]
    selected = X_sub.columns[coefs != 0]

    for feat in selected:
        counter[feat] += 1

stable_pool_df = pd.DataFrame({
    "metabolite": list(counter.keys()),
    "count": list(counter.values())
}).sort_values("count", ascending=False)

stable_pool_df["frequency"] = stable_pool_df["count"] / 200
stable_pool_df["clean_name"] = stable_pool_df["metabolite"].str.replace("metab__", "", regex=False)

display(stable_pool_df.head(25)[["clean_name", "count", "frequency"]])

# use top 15 candidates as candidate pool
candidate_pool = stable_pool_df.head(15)["metabolite"].tolist()

# optional: remove unknown X metabolites
candidate_pool = [
    m for m in candidate_pool
    if not m.replace("metab__", "").startswith("X-")
]

print("Candidate pool size:", len(candidate_pool))

# =========================
# 3. Search all 5-marker subsets inside candidate pool
# using CV on TRAIN ONLY
# =========================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

subset_results = []

base_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=5000))
])

for subset in combinations(candidate_pool, 5):
    aucs = cross_val_score(
        base_model,
        X_train[list(subset)],
        y_train,
        cv=cv,
        scoring="roc_auc"
    )

    subset_results.append({
        "subset": subset,
        "cv_mean_auc": aucs.mean(),
        "cv_std_auc": aucs.std()
    })

subset_df = pd.DataFrame(subset_results).sort_values(
    "cv_mean_auc",
    ascending=False
)

display(subset_df.head(10))

best5_panel = list(subset_df.iloc[0]["subset"])

print("\nBest train-CV 5-marker panel:")
for m in best5_panel:
    print(m.replace("metab__", ""))

# =========================
# 4. Evaluate best 5 on held-out test ONCE
# =========================

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=5000))
])

final_model.fit(X_train[best5_panel], y_train)

probs = final_model.predict_proba(X_test[best5_panel])[:, 1]

print("\nHeld-out AUC:", roc_auc_score(y_test, probs))

# =========================
# 5. Compare whether yesterday panel was available/ranked
# =========================

yesterday_panel = [
    "metab__1-stearoyl-2-docosahexaenoyl-GPC (18:0/22:6)",
    "metab__3-(4-hydroxyphenyl)lactate",
    "metab__N-acetylglutamate",
    "metab__N-palmitoyl-sphinganine (d18:0/16:0)",
    "metab__N-palmitoyl-sphingosine (d18:1/16:0)"
]

yesterday_set = set(yesterday_panel)

subset_df["is_yesterday_panel"] = subset_df["subset"].apply(
    lambda s: set(s) == yesterday_set
)

print("\nYesterday panel in candidate pool?")
print(all(m in candidate_pool for m in yesterday_panel))

if subset_df["is_yesterday_panel"].any():
    display(subset_df[subset_df["is_yesterday_panel"]])
else:
    print("Yesterday panel was not one of the searched subsets.")

,clean_name,count,frequency
2,1-methylurate,181,0.905
0,N-acetylglutamate,176,0.880
5,N-palmitoyl-sphingosine (d18:1/16:0),168,0.840
16,N-palmitoyl-sphinganine (d18:0/16:0),45,0.225
12,1-stearoyl-2-arachidonoyl-GPC (18:0/20:4),42,0.210
15,3-(4-hydroxyphenyl)lactate,42,0.210
20,N4-acetylcytidine,40,0.200
7,"sphingomyelin (d18:1/18:1, d18:2/18:0)",40,0.200
4,N-stearoyl-sphinganine (d18:0/18:0)*,37,0.185
6,phenyllactate (PLA),35,0.175


Candidate pool size: 15


,subset,cv_mean_auc,cv_std_auc
127,"(metab__1-methylurate, metab__N-acetylglutamat...",0.870818,0.024192
27,"(metab__1-methylurate, metab__N-acetylglutamat...",0.868010,0.015280
184,"(metab__1-methylurate, metab__N-acetylglutamat...",0.866361,0.018212
171,"(metab__1-methylurate, metab__N-acetylglutamat...",0.865201,0.025643
391,"(metab__1-methylurate, metab__N-palmitoyl-sphi...",0.859829,0.041278
227,"(metab__1-methylurate, metab__N-acetylglutamat...",0.859524,0.036766
199,"(metab__1-methylurate, metab__N-acetylglutamat...",0.859219,0.015259
247,"(metab__1-methylurate, metab__N-acetylglutamat...",0.859096,0.067172
64,"(metab__1-methylurate, metab__N-acetylglutamat...",0.858425,0.033884
82,"(metab__1-methylurate, metab__N-acetylglutamat...",0.857082,0.011203



Best train-CV 5-marker panel:
1-methylurate
N-acetylglutamate
1-stearoyl-2-arachidonoyl-GPC (18:0/20:4)
3-(4-hydroxyphenyl)lactate
p-cresol glucuronide*

Held-out AUC: 0.7524038461538461

Yesterday panel in candidate pool?
False
Yesterday panel was not one of the searched subsets.


In [32]:
from collections import Counter
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score

# use metabolomics only
X_metab = X_all[[c for c in X_all.columns if c.startswith("metab__")]]

def stability_select(X_train, y_train, k, n_inner=100, C=0.06, seed=42):
    counter = Counter()

    sss = StratifiedShuffleSplit(
        n_splits=n_inner,
        test_size=0.2,
        random_state=seed
    )

    for idx, _ in sss.split(X_train, y_train):
        X_sub = X_train.iloc[idx]
        y_sub = y_train.iloc[idx]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("lasso", LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=C,
                max_iter=5000,
                random_state=seed
            ))
        ])

        model.fit(X_sub, y_sub)

        coefs = model.named_steps["lasso"].coef_[0]
        selected = X_sub.columns[coefs != 0]

        for feat in selected:
            counter[feat] += 1

    ranked = pd.DataFrame({
        "feature": list(counter.keys()),
        "count": list(counter.values())
    }).sort_values("count", ascending=False)

    return ranked.head(k)["feature"].tolist(), ranked


all_rows = []
panel_counts_by_k = {k: Counter() for k in range(4, 31)}

for k in range(4, 31):
    aucs = []
    accs = []
    bal_accs = []

    for seed in range(50):   # change to 100 later if you want
        X_train, X_test, y_train, y_test = train_test_split(
            X_metab,
            y,
            test_size=0.2,
            stratify=y,
            random_state=seed
        )

        panel, ranked = stability_select(
            X_train,
            y_train,
            k=k,
            n_inner=100,
            C=0.06,
            seed=seed
        )

        for feat in panel:
            panel_counts_by_k[k][feat] += 1

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("logreg", LogisticRegression(max_iter=5000))
        ])

        model.fit(X_train[panel], y_train)

        probs = model.predict_proba(X_test[panel])[:, 1]
        preds = model.predict(X_test[panel])

        aucs.append(roc_auc_score(y_test, probs))
        accs.append(accuracy_score(y_test, preds))
        bal_accs.append(balanced_accuracy_score(y_test, preds))

    all_rows.append({
        "k": k,
        "mean_auc": np.mean(aucs),
        "median_auc": np.median(aucs),
        "std_auc": np.std(aucs),
        "min_auc": np.min(aucs),
        "max_auc": np.max(aucs),
        "mean_accuracy": np.mean(accs),
        "mean_balanced_accuracy": np.mean(bal_accs)
    })

size_results = pd.DataFrame(all_rows).sort_values("mean_auc", ascending=False)

display(size_results)

KeyboardInterrupt: 

In [ ]:
best_k = int(size_results.iloc[0]["k"])
print("Best k:", best_k)

final_panel_counts = pd.DataFrame({
    "metabolite": list(panel_counts_by_k[best_k].keys()),
    "count": list(panel_counts_by_k[best_k].values())
}).sort_values("count", ascending=False)

final_panel_counts["frequency"] = final_panel_counts["count"] / 50
final_panel_counts["clean_name"] = final_panel_counts["metabolite"].str.replace("metab__", "", regex=False)

display(final_panel_counts.head(best_k)[["clean_name", "count", "frequency"]])